> **Note**  
> [More Cultural Names](https://github.com/hmlendea/more-cultural-names) (MCN) is used here strictly as an **evaluation oracle**: its attested exonyms serve as a gold standard against which NTE's output is measured. MCN data has not been used to populate, enrich, or tune NTE's own lookup database, which is built independently from [GeoNames](https://www.geonames.org/) alternate-name dumps and [Wikidata](https://www.wikidata.org/) CC0 snapshots. Any agreement between NTE output and MCN entries therefore reflects genuine coverage by NTE's pipeline, not data leakage.
>
> MCN is distributed under the [GPL-3.0 License](https://github.com/hmlendea/more-cultural-names/blob/master/LICENSE.md). This notebook does not redistribute MCN data; it reads the repository in-place from `data/repos/more-cultural-names/`.

In [ ]:
import sqlite3
import collections
import pandas as pd
import xml.etree.ElementTree as ET

from pathlib import Path
from dataclasses import dataclass
from enum import Enum

pd.set_option("display.max_rows", 1000)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

PROJECT_ROOT  = Path.cwd().parent
MCN_PATH      = PROJECT_ROOT / "data" / "repos" / "more-cultural-names"
MCN_LOCATIONS = MCN_PATH / "locations.xml"
MCN_LANGUAGES = MCN_PATH / "languages.xml"
DB_PATH       = PROJECT_ROOT / "data" / "names.sqlite"

HIT = ("exact", "case_fold")

In [ ]:
def get_conn() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn

def run_query(sql: str, params: tuple = ()) -> pd.DataFrame:
    with get_conn() as conn:
        return pd.read_sql_query(sql, conn, params=params)

In [ ]:
# Build the MCN-language -> ISO-code map from languages.xml
languages_tree = ET.parse(MCN_LANGUAGES)

@dataclass(frozen=True)
class LangInfo:
    mcn_id:  str
    iso1:    str | None   # iso-639-1
    iso2:    str | None   # iso-639-2
    iso3:    str | None   # iso-639-3
    fallbacks: tuple[str, ...] = ()

    @property
    def iso_codes(self) -> tuple[str, ...]:
        return tuple(dict.fromkeys(c for c in (self.iso1, self.iso2, self.iso3) if c))

    @property
    def has_iso(self) -> bool:
        return bool(self.iso_codes)

lang_info: dict[str, LangInfo] = {}
for lang in languages_tree.getroot().findall("Language"):
    mcn_id = (lang.findtext("Id") or "").strip()
    if not mcn_id:
        continue
    code = lang.find("Code")
    iso1 = iso2 = iso3 = None
    if code is not None:
        iso1 = code.attrib.get("iso-639-1") or None
        iso2 = code.attrib.get("iso-639-2") or None
        iso3 = code.attrib.get("iso-639-3") or None
    fb = tuple(
        (e.text or "").strip()
        for e in lang.findall("FallbackLanguages/LanguageId")
        if (e.text or "").strip()
    )
    lang_info[mcn_id] = LangInfo(mcn_id, iso1, iso2, iso3, fb)

n_total   = len(lang_info)
n_iso     = sum(1 for li in lang_info.values() if li.has_iso)
print(f"MCN languages defined          : {n_total}")
print(f"  - with at least one ISO code : {n_iso} ({n_iso/n_total:.1%})")
print(f"  - without any ISO code       : {n_total - n_iso} ({(n_total-n_iso)/n_total:.1%})")

In [ ]:
locations_tree = ET.parse(MCN_LOCATIONS)

# (title, geonames_id) -> {mcn_language_id: name_value}
ck3_names: dict[tuple[str, int], dict[str, str]] = {}
for loc in locations_tree.iter("LocationEntity"):
    geonames_tag = loc.find("GeoNamesId")
    if geonames_tag is None:
        continue
    geonames_id = int(geonames_tag.text)

    names = {
        n.attrib["language"]: n.attrib["value"]
        for n in loc.findall("Names/Name")
    }
    if not names:
        continue

    for game_id in loc.findall("GameIds/GameId"):
        if game_id.attrib.get("game") != "CK3":
            continue
        title = game_id.text.strip() if game_id.text is not None else ""
        if title.startswith("d_nf") or title.startswith("b_"):
            continue
        ck3_names[(title, geonames_id)] = names

ck3_geonames_ids = sorted({gid for _, gid in ck3_names})
print(f"CK3 (title, geonames_id) pairs : {len(ck3_names)}")
print(f"distinct GeoNames ids          : {len(ck3_geonames_ids)}")

attested_rows = [
    (title, gid, mcn_lang, name)
    for (title, gid), names in ck3_names.items()
    for mcn_lang, name in names.items()
]
print(f"attested (title, language) rows: {len(attested_rows)}")
print(f"distinct MCN languages used    : {len({r[2] for r in attested_rows})}")

In [ ]:
@dataclass(frozen=True)
class NameCandidate:
    source:      str          # 'geonames' / 'wikidata'
    source_lang: str          # GeoNames isolanguage, or Wikidata geo_lang/wd_lang
    name:        str
    qid:         str | None = None

def get_nte_names_for_geonames_id(geonames_id: int) -> list[NameCandidate]:
    df = run_query("""
        SELECT
            'geonames'      AS source,
            NULL            AS qid,
            isolanguage     AS source_lang,
            alternate_name  AS name
        FROM alternate_name
        WHERE geonameid = ?

        UNION ALL

        SELECT
            'wikidata'      AS source,
            n.qid           AS qid,
            COALESCE(n.geo_lang, n.wd_lang, '') AS source_lang,
            n.name          AS name
        FROM wikidata_location_geonames AS g
        JOIN wikidata_location_name     AS n
          ON n.qid = g.qid
        WHERE g.geonames_id = CAST(? AS TEXT)

        ORDER BY source, source_lang, name
    """, (geonames_id, geonames_id))

    return [
        NameCandidate(
            source=row.source,
            source_lang=row.source_lang or "",
            name=row.name,
            qid=row.qid,
        )
        for row in df.itertuples(index=False)
    ]

nte_cache: dict[int, list[NameCandidate]] = {
    gid: get_nte_names_for_geonames_id(gid)
    for gid in ck3_geonames_ids
}

n_empty = sum(1 for c in nte_cache.values() if not c)
print(f"GeoNames ids resolved          : {len(nte_cache)}")
print(f"  - with >=1 NTE candidate     : {len(nte_cache) - n_empty}")
print(f"  - with no NTE candidate      : {n_empty}")
print(f"total NTE candidate rows cached: {sum(len(c) for c in nte_cache.values())}")

In [ ]:
class MatchStatus(Enum):
    EXACT     = "exact"      # MCN name appears verbatim in NTE (any source / tag)
    CASE_FOLD = "case_fold"  # appears after case folding
    DB_EMPTY  = "db_empty"   # geonames id has no NTE candidates at all
    DB_MISS   = "db_miss"    # candidates exist, but MCN name not among them

@dataclass
class Result:
    title:           str
    geonames_id:     int
    mcn_lang:        str
    mcn_name:        str
    status:          MatchStatus
    matched_sources: tuple[str, ...] = ()   # ('geonames',) / ('wikidata',) / both
    matched_langs:   tuple[str, ...] = ()   # 'geonames:la', 'wikidata:la', ...
    tag_agree:       bool = False           # a matching row carried this language's ISO code

def _uniq(values):
    return tuple(dict.fromkeys(v for v in values if v is not None))

def compare_one(title: str, geonames_id: int, mcn_lang: str, mcn_name: str) -> Result:
    candidates = nte_cache.get(geonames_id, [])
    if not candidates:
        return Result(title, geonames_id, mcn_lang, mcn_name, MatchStatus.DB_EMPTY)

    iso_codes = set(lang_info.get(mcn_lang, LangInfo(mcn_lang, None, None, None)).iso_codes)

    def finalize(rows, status):
        matched_sources = tuple(sorted({c.source for c in rows}))
        matched_langs   = _uniq(f"{c.source}:{c.source_lang}" for c in rows)
        tag_agree       = bool(iso_codes) and any(c.source_lang in iso_codes for c in rows)
        return Result(title, geonames_id, mcn_lang, mcn_name, status,
                      matched_sources, matched_langs, tag_agree)

    exact = [c for c in candidates if c.name == mcn_name]
    if exact:
        return finalize(exact, MatchStatus.EXACT)

    folded = [c for c in candidates if c.name.lower() == mcn_name.lower()]
    if folded:
        return finalize(folded, MatchStatus.CASE_FOLD)

    return Result(title, geonames_id, mcn_lang, mcn_name, MatchStatus.DB_MISS)

results = [compare_one(*row) for row in attested_rows]
print(f"comparisons run: {len(results)}")

# Global summary

Aggregate match status over all attested MCN CK3 rows. `hit_rate` = (exact + case_fold) / all rows. `recall_present` = hits / rows whose GeoNames id has at least one NTE candidate - i.e. how often NTE has the right name given that it knows the place at all

In [ ]:
global_counts = collections.Counter(r.status for r in results)
n_all   = len(results)
n_hit   = sum(global_counts[s] for s in (MatchStatus.EXACT, MatchStatus.CASE_FOLD))
n_empty = global_counts[MatchStatus.DB_EMPTY]
n_present = n_all - n_empty

print("Status breakdown (all attested rows):")
for status in MatchStatus:
    n = global_counts[status]
    print(f"  {status.value:11s} {n:6d}  ({n/n_all:.1%})")

print()
print(f"hit_rate        (hits / all rows)              : {n_hit}/{n_all} = {n_hit/n_all:.1%}")
print(f"recall_present  (hits / rows w/ NTE candidates): {n_hit}/{n_present} = {n_hit/n_present:.1%}")

# Source attribution among hits
src_buckets = collections.Counter(
    "+".join(r.matched_sources)
    for r in results
    if r.status.value in HIT
)
print()
print("Hit source attribution:")
for bucket, n in src_buckets.most_common():
    print(f"  {bucket:20s} {n:6d}  ({n/n_hit:.1%})")

# How often the matching NTE row actually carried this language's ISO tag
n_tag_agree = sum(1 for r in results if r.status.value in HIT and r.tag_agree)
print()
print(f"hits where a matching row carried the MCN language's ISO tag: "
      f"{n_tag_agree}/{n_hit} = {n_tag_agree/n_hit:.1%}")
print("(the remainder are correct *strings* found under a different/blank language tag)")

## Per-language report

One row per MCN language, sorted by number of attested names (descending). Columns:

- **attested** — MCN names for this language across CK3 titles.
- **has_iso** — whether MCN assigns this language an ISO code (if not, GeoNames tag-matching is impossible by construction).
- **exact / case_fold / db_miss / db_empty** — status counts.
- **hits** — exact + case_fold.
- **hit_rate** — hits / attested.
- **recall_present** — hits / (attested − db_empty).
- **via_wikidata** — hits that Wikidata contributed (alone or jointly with GeoNames).

In [ ]:
by_lang: dict[str, list[Result]] = collections.defaultdict(list)
for r in results:
    by_lang[r.mcn_lang].append(r)

rows = []
for mcn_lang, rs in by_lang.items():
    c = collections.Counter(r.status for r in rs)
    attested = len(rs)
    exact     = c[MatchStatus.EXACT]
    case_fold = c[MatchStatus.CASE_FOLD]
    db_miss   = c[MatchStatus.DB_MISS]
    db_empty  = c[MatchStatus.DB_EMPTY]
    hits      = exact + case_fold
    present   = attested - db_empty
    via_wd    = sum(
        1 for r in rs
        if r.status.value in HIT and "wikidata" in r.matched_sources
    )
    li = lang_info.get(mcn_lang)
    rows.append({
        "mcn_lang":       mcn_lang,
        "iso":            (li.iso1 or li.iso3) if (li and li.has_iso) else "",
        "has_iso":        bool(li and li.has_iso),
        "attested":       attested,
        "exact":          exact,
        "case_fold":      case_fold,
        "db_miss":        db_miss,
        "db_empty":       db_empty,
        "hits":           hits,
        "hit_rate":       hits / attested if attested else 0.0,
        "recall_present": hits / present if present else float("nan"),
        "via_wikidata":   via_wd,
    })

per_lang_df = (
    pd.DataFrame(rows)
    .sort_values(["attested", "hits"], ascending=[False, False])
    .reset_index(drop=True)
)

# Pretty percentage columns for display
def _fmt(df):
    out = df.copy()
    out["hit_rate"]       = out["hit_rate"].map(lambda x: f"{x:.1%}")
    out["recall_present"] = out["recall_present"].map(
        lambda x: "—" if pd.isna(x) else f"{x:.1%}")
    return out

print(f"distinct MCN languages benchmarked: {len(per_lang_df)}")
display(_fmt(per_lang_df))

### Languages with the most attested names

The long tail of 800+ languages is dominated by a handful of high-volume ones. This view focuses on the languages where the benchmark is statistically meaningful (most attested names).

In [ ]:
TOP_N = 40
top = per_lang_df.head(TOP_N)
display(_fmt(top))

print(f"Coverage of these top {TOP_N} languages:")
print(f"  attested rows : {top['attested'].sum():6d}  "
      f"({top['attested'].sum()/n_all:.1%} of all attested)")
print(f"  hits          : {top['hits'].sum():6d}")
print(f"  hit_rate      : {top['hits'].sum()/top['attested'].sum():.1%}")

## Effect of ISO-code availability

GeoNames tags alternate names by ISO `isolanguage`. MCN languages that carry **no** ISO code (historical / liturgical / reconstructed varieties) can still produce string hits — when the exonym happens to coincide with a name stored under some other tag — but they are structurally disadvantaged. Splitting hit rate by ISO availability quantifies that gap.

In [ ]:
iso_split = []
for has_iso, sub in per_lang_df.groupby("has_iso"):
    attested = sub["attested"].sum()
    hits     = sub["hits"].sum()
    empty    = sub["db_empty"].sum()
    present  = attested - empty
    iso_split.append({
        "has_iso":        bool(has_iso),
        "languages":      len(sub),
        "attested":       int(attested),
        "hits":           int(hits),
        "hit_rate":       f"{hits/attested:.1%}" if attested else "—",
        "recall_present": f"{hits/present:.1%}" if present else "—",
    })

display(pd.DataFrame(iso_split).sort_values("has_iso", ascending=False))

## Distribution of per-language hit rates

Among languages with a reasonable sample size (≥ `MIN_ATTESTED` attested names), how is hit rate distributed? This shows whether coverage is broadly even or concentrated in a few well-served languages.

In [ ]:
MIN_ATTESTED = 20
sample = per_lang_df[per_lang_df["attested"] >= MIN_ATTESTED].copy()

print(f"Languages with >= {MIN_ATTESTED} attested names: {len(sample)}")
print()
print("Per-language hit_rate distribution:")
print(sample["hit_rate"].describe().to_string())

bins = [0, .05, .10, .20, .30, .50, .75, 1.01]
labels = ["0-5%", "5-10%", "10-20%", "20-30%", "30-50%", "50-75%", "75-100%"]
sample["band"] = pd.cut(sample["hit_rate"], bins=bins, labels=labels, right=False)
band_counts = sample["band"].value_counts().reindex(labels).fillna(0).astype(int)

print()
print("Languages per hit_rate band:")
for label, cnt in band_counts.items():
    bar = "#" * cnt
    print(f"  {label:8s} {cnt:3d}  {bar}")

## Where misses come from

A miss is either `db_empty` (NTE has no alternate names for that GeoNames id at all — a *place-coverage* gap) or `db_miss` (NTE knows the place but lacks that specific exonym — a *name-coverage* gap)

In [ ]:
miss_rows = [r for r in results if r.status.value not in HIT]
miss_counts = collections.Counter(r.status for r in miss_rows)
n_miss = len(miss_rows)

print(f"Total misses: {n_miss} ({n_miss/n_all:.1%} of attested rows)")
for status in (MatchStatus.DB_MISS, MatchStatus.DB_EMPTY):
    n = miss_counts[status]
    print(f"  {status.value:9s} {n:6d}  ({n/n_miss:.1%} of misses)")

# db_empty is a property of the GeoNames id, not the language: report distinct ids.
empty_ids = sorted({r.geonames_id for r in results if r.status == MatchStatus.DB_EMPTY})
print()
print(f"distinct GeoNames ids with NO NTE candidates: {len(empty_ids)} "
      f"/ {len(ck3_geonames_ids)} "
      f"({len(empty_ids)/len(ck3_geonames_ids):.1%} of benchmarked places)")